# stream 与 astream：同步 / 异步流式执行

`invoke` 要等整图跑完才一次性返回；`stream` / `astream` 则**边执行边产出**——
每跑完一个超步（节点）就吐出一次该节点的更新（默认 `stream_mode="updates"`），
适合实时展示执行进度、尽早响应用户。二者 API 完全同构，只差同步/异步：

| | stream | astream |
|---|---|---|
| 类型 | 同步方法，返回普通迭代器 | 异步方法，返回异步迭代器 |
| 消费方式 | `for chunk in graph.stream(...)` | `async for chunk in graph.astream(...)` |
| 线程模型 | **阻塞**当前线程直到流结束 | 协程挂起等待，**不阻塞事件循环** |
| 适用场景 | 脚本、Notebook 顺序探索 | FastAPI 等异步 Web 服务、单事件循环并发多会话 |

下面用最小图实测：3 个流同时跑，`astream` 并发总耗时 ≈ 单次；`stream` 串行 = 3 倍。

In [1]:
import time
from typing import Annotated, TypedDict

import operator
from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    log: Annotated[list[str], operator.add]


def node_a(state: State) -> dict:
    time.sleep(1)                       # 模拟耗时（如调 LLM）
    return {"log": ["A 完成"]}


def node_b(state: State) -> dict:
    time.sleep(1)
    return {"log": ["B 完成"]}


builder = StateGraph(state_schema=State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)
graph = builder.compile()

# stream：每跑完一个节点就产出一个 chunk，key 是节点名
print("--- stream 逐块输出 ---")
for chunk in graph.stream({"log": []}):
    print(chunk)

# 串行跑 3 次：每次约 2s，总计约 6s——stream 阻塞当前线程，只能排队
t0 = time.perf_counter()
for _ in range(3):
    for _ in graph.stream({"log": []}):
        pass
print(f"\n3 次 stream 串行总耗时: {time.perf_counter() - t0:.1f}s（约 6s）")

--- stream 逐块输出 ---
{'node_a': {'log': ['A 完成']}}
{'node_b': {'log': ['B 完成']}}

3 次 stream 串行总耗时: 6.0s（约 6s）


In [ ]:
import asyncio

from typing import Annotated, TypedDict

import operator
from langgraph.graph import END, START, StateGraph


class AState(TypedDict):
    log: Annotated[list[str], operator.add]


# 异步节点：配 astream / ainvoke 使用
async def a_node_a(state: AState) -> dict:
    await asyncio.sleep(1)
    return {"log": ["A 完成"]}


async def a_node_b(state: AState) -> dict:
    await asyncio.sleep(1)
    return {"log": ["B 完成"]}


a_builder = StateGraph(state_schema=AState)
a_builder.add_node("node_a", a_node_a)
a_builder.add_node("node_b", a_node_b)
a_builder.add_edge(START, "node_a")
a_builder.add_edge("node_a", "node_b")
a_builder.add_edge("node_b", END)
agraph = a_builder.compile()


async def run_once(i: int) -> None:
    async for chunk in agraph.astream({"log": []}):   # 消费方式：async for
        print(f"流{i}: {chunk}")


# 3 个 astream 并发：挂起等待不阻塞事件循环，总耗时 ≈ 单次（约 2s）
async def main():
    t0 = time.perf_counter()
    await asyncio.gather(*(run_once(i) for i in range(3)))
    print(f"\n3 个 astream 并发总耗时: {time.perf_counter() - t0:.1f}s（约 2s）")


await main()   # Notebook 顶层可直接 await；脚本里用 asyncio.run(main())

## stream 与 astream 的区别（实测结论）

1. **消费方式**：`stream` 用普通 `for`；`astream` 必须在事件循环里 `async for`（Notebook 顶层可直接 `await`，脚本用 `asyncio.run()` 包一层）。
2. **线程模型（本质区别）**：`stream` 阻塞当前线程，循环期间什么都干不了；`astream` 在等待图执行时挂起协程，同一事件循环还能跑别的请求——上面对比：并发 3 个流，astream 约 2s，stream 串行约 6s。
3. **其余 API 完全一致**：入参、`config`、`stream_mode`（updates/values/messages/custom 等）两边通用，会一个就会另一个。
4. **配套约定**：异步图（含 `async def` 节点）要用 `astream`/`ainvoke` 消费；同步图用 `stream`/`invoke`。异步 Web 服务（FastAPI 等）里选 `astream`——用 `stream` 会卡住整个服务的事件循环。